# P146 — Aprendizaje eficiente en comunicación de redes profundas con datos descentralizados

## 1. Título y paper

**Paper:** *Communication-Efficient Learning of Deep Networks from Decentralized Data*  
**Autoría:** H. Brendan McMahan, Eider Moore, Daniel Ramage, Seth Hampson, Blaise Agüera y Arcas  
**Año y venue:** 2017 · AISTATS 2017, PMLR 54, 1273–1282  
**Nivel:** L2 · **Motor:** `federado`  
**Ficha completa:** [`P146_federado`](../../papers/foundational/P146_federado/README.md)

**Hito:** Entrena un modelo compartido sin que los datos salgan del dispositivo, promediando modelos en vez de recoger registros.

- [arXiv:1602.05629](https://arxiv.org/abs/1602.05629)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los datos más útiles para entrenar —lo que se escribe en el teclado, lo que se fotografía— son los más sensibles y viven en millones de dispositivos con conexión lenta e intermitente. Centralizarlos es caro en comunicación y problemático en privacidad.
2. Ejecutar una implementación mínima de la propuesta: Promediado federado: cada cliente entrena varias épocas en local sobre sus propios datos y envía solo los pesos resultantes; el servidor los promedia y devuelve el modelo. Más cómputo local a cambio de menos rondas de comunicación.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P143


## 4. Intuición

Los datos más útiles para entrenar son los más sensibles y viven en millones de dispositivos. Promediar **modelos** en vez de recoger **registros** cambia por completo la superficie del problema.


## 5. Concepto mínimo

```text
Ronda t:
  1. el servidor manda el modelo actual a los clientes
  2. cada cliente entrena E épocas en LOCAL sobre sus datos
  3. cada cliente devuelve solo los PESOS
  4. el servidor promedia
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('federado', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿A cuánto llega la exactitud sin que salga ningún dato?
2. ¿Qué aporta más cómputo local?
3. ¿Rompe la heterogeneidad el promediado?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('federado', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('federado', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

**0,993** en 12 rondas con 5 épocas locales, sin transmitir un solo registro. Más cómputo local acelera: en la ronda 1 se va de 0,975 con una época a 0,985 con cinco. Y con cada cliente viendo **una sola clase**, la exactitud final es 0,988 frente a 0,993 — apenas 0,005.


## 10. Comentario pedagógico

Ese último resultado hay que leerlo con cuidado y el motor lo dice: **esta maqueta NO reproduce el fallo por heterogeneidad** que documenta la literatura. Con 20 clientes repartidos simétricamente y un modelo lineal, promediar cancela los sesgos locales. El fallo real aparece con redes profundas, participación desigual y clientes que no se compensan.


## 11. Error o anti-patrón deliberado

Anti-patrón: presentar el aprendizaje federado como privacidad.


In [ ]:
print('Que los datos no salgan no significa que no se filtren.')
print('De los gradientes se pueden reconstruir ejemplos de entrenamiento.')
print('Federado + privacidad diferencial + agregacion segura: las tres, no una.')

## 12. Corrección

Las cuatro configuraciones:


In [ ]:
r = run_paper_lab('federado', seed=3)['result']
for f in r['resultados']:
    print(f"{f['distribucion']:>22} | {f['epocas_locales']} epocas -> {f['exactitud_final']}")
for c in r['coste_de_comunicacion']:
    print(c)

## 13. Desafío guiado

Explica por qué la comunicación es el recurso caro en este escenario y el cómputo no, y qué consecuencia tiene eso en el diseño del algoritmo.


In [ ]:
r = run_paper_lab('federado', seed=3)['result']
show(r)

## 14. Desafío autónomo

Identifica datos de tu organización que no se pueden centralizar por normativa. Estima si el promediado federado sería viable con su volumen y su conectividad.


## 15. Evidencia de aprendizaje

Guarda la estimación y el obstáculo principal.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P146_federado/README.md) · evaluación formal: [`assessments/papers/P146_federado.md`](../../assessments/papers/P146_federado.md)


## 16. Cierre

Queda la frontera: modelos que no solo aprenden del mundo sino que lo imaginan.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
